In [17]:
!pip install sentence-transformers faiss-cpu networkx -q pandas

In [1]:
import json
import gc
import os
import pickle
import itertools
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd
import faiss
import networkx as nx
import torch
import torch.nn.functional as F
from sentence_transformers import CrossEncoder
from tqdm.auto import tqdm

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"PyTorch : {torch.__version__}")
print(f"FAISS   : {faiss.__version__}")
print(f"Device  : {DEVICE}")

PyTorch : 2.4.1+cu124
FAISS   : 1.13.2
Device  : cuda


## 1. Configuration

In [ ]:
DATA_DIR   = Path('workspace/embeddings')
OUTPUT_DIR = Path('./retrieval_rq3')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

BDC_CHUNKS_JSON     = DATA_DIR / 'VD_bdc_chunks.json'
PSC_CHUNKS_JSON     = DATA_DIR / 'VD_psc_chunks.json'
GRAPHML_PATH        = DATA_DIR / 'bdc_graph.graphml'
VALIDATION_SET_JSON = DATA_DIR / 'ir_ground_truth.json'

token = os.getenv("HF_TOKEN")

# Precomputed embeddings
PSC_EMBEDDINGS_NPY = DATA_DIR / 'psc_embeddings_e5-intertext.npy'
PSC_IDS_JSON       = DATA_DIR / 'psc_ids_e5-intertext.json'
BDC_EMBEDDINGS_NPY = DATA_DIR / 'bdc_embeddings_e5.npy'
BDC_IDS_JSON       = DATA_DIR / 'bdc_ids_e5.json'

# Precomputed full BDC→PSC retrieval PKL
ALL_BDC_RETRIEVAL_PKL = DATA_DIR / 'all_bdc_retrieval_e5_top100.pkl'

CROSS_ENCODER_MODEL = 'julian-schelb/PhilBerta-class-latin-intertext-v1'

# Hyperparameters, weill get updated with sweep
N_NEIGHBOURS         = 1      # graph neighbours per query letter
SIM_THRESHOLD        = 0.8    # min cosine sim between query chunk and neighbour chunk
TOP_N_FROM_NEIGHBOUR = 20     # PSC candidates taken from each qualifying neighbour
RETRIEVAL_TOP_K      = 100    # bi-encoder candidates; CE input cap
FINAL_K              = 20     # candidates kept after CE reranking
RRF_K                = 60     # DO NOT CHANGE
RERANK_BATCH         = 256

print('Configuration loaded.')

Configuration loaded.


## 2. Load data

In [3]:
def load_chunks(path):
    """Return {chunk_id: text} from BDC or PSC chunk JSON."""
    with open(path) as f:
        data = json.load(f)
    chunks = data['chunks'] if isinstance(data, dict) and 'chunks' in data else data
    return {c['chunk_id']: c['text'] for c in chunks}

print('Loading chunk texts...')
bdc_text_lookup = load_chunks(BDC_CHUNKS_JSON)
psc_text_lookup = load_chunks(PSC_CHUNKS_JSON)
print(f'  BDC chunks : {len(bdc_text_lookup):,}')
print(f'  PSC chunks : {len(psc_text_lookup):,}')

Loading chunk texts...
  BDC chunks : 19,466
  PSC chunks : 46,408


In [ ]:
print('Loading embeddings...')
psc_embeddings = np.load(PSC_EMBEDDINGS_NPY)
psc_ids        = json.loads(PSC_IDS_JSON.read_text())
bdc_embeddings = np.load(BDC_EMBEDDINGS_NPY)
bdc_ids        = json.loads(BDC_IDS_JSON.read_text())
print(f'  PSC embeddings : {psc_embeddings.shape}')
print(f'  BDC embeddings : {bdc_embeddings.shape}')
assert psc_embeddings.shape[1] == bdc_embeddings.shape[1], 'Embedding dim mismatch!'

# chunk_id → embedding index
bdc_id_to_idx = {cid: idx for idx, cid in enumerate(bdc_ids)}

Loading embeddings...
  PSC embeddings : (46408, 1024)
  BDC embeddings : (961431, 1024)


In [5]:
print('Loading epistolary graph...')
G = nx.read_graphml(GRAPHML_PATH)
print(f'  Nodes    : {G.number_of_nodes():,}')
print(f'  Edges    : {G.number_of_edges():,}')
print(f'  Isolated : {sum(1 for n in G.nodes if G.degree(n) == 0):,}')

Loading epistolary graph...
  Nodes    : 13,114
  Edges    : 42,458
  Isolated : 3,420


In [6]:
print('Loading validation set...')
with open(VALIDATION_SET_JSON) as f:
    validation_set = json.load(f)

ground_truth = validation_set['ground_truth']
query_ids    = [e['query_chunk_id'] for e in ground_truth]
print(f'  Total entries  : {len(ground_truth)}')
print(f'  Explicit       : {sum(1 for e in ground_truth if e["reference_type"] == "explicit")}')
print(f'  Implicit       : {sum(1 for e in ground_truth if e["reference_type"] == "implicit")}')

Loading validation set...
  Total entries  : 100
  Explicit       : 50
  Implicit       : 50


## 3. Stage 1 — Bi-encoder retrieval (precomputed)

Load the precomputed full BDC→PSC top-100 retrieval PKL.  
If missing, compute from precomputed embeddings via batched FAISS 

In [7]:
print('Building FAISS IndexFlatIP over PSC...')
faiss_index = faiss.IndexFlatIP(psc_embeddings.shape[1])
faiss_index.add(psc_embeddings.astype('float32'))
print(f'  Vectors in index : {faiss_index.ntotal:,}')

Building FAISS IndexFlatIP over PSC...
  Vectors in index : 46,408


In [8]:
CHECKPOINT_PKL         = ALL_BDC_RETRIEVAL_PKL.parent / (ALL_BDC_RETRIEVAL_PKL.stem + '_checkpoint.pkl')
BATCH                  = 4096
CHECKPOINT_EVERY_N     = 10


def safe_load_pickle(path):
    if not path.exists():
        return None
    if path.stat().st_size == 0:
        print(f'Empty file: {path} → deleting')
        path.unlink()
        return None
    try:
        with open(path, 'rb') as f:
            return pickle.load(f)
    except (EOFError, pickle.UnpicklingError):
        print(f'Corrupted pickle: {path} → deleting')
        path.unlink()
        return None


def atomic_pickle_dump(obj, path):
    tmp = path.with_suffix('.tmp')
    with open(tmp, 'wb') as f:
        pickle.dump(obj, f, protocol=pickle.HIGHEST_PROTOCOL)
    tmp.replace(path)


# Try loading final result first
all_bdc_retrieval = safe_load_pickle(ALL_BDC_RETRIEVAL_PKL)

if all_bdc_retrieval is not None:
    print(f'Loaded full retrieval: {len(all_bdc_retrieval):,} entries')

else:
    # Try checkpoint
    all_bdc_retrieval = safe_load_pickle(CHECKPOINT_PKL) or {}
    completed   = len(all_bdc_retrieval)
    start_batch = (completed // BATCH) * BATCH
    print(f'Resuming from {completed:,} / {len(bdc_ids):,}' if completed else 'Starting fresh')

    for batch_num, start in enumerate(tqdm(range(start_batch, len(bdc_ids), BATCH), desc='FAISS search')):
        end        = min(start + BATCH, len(bdc_ids))
        batch_embs = bdc_embeddings[start:end].astype('float32', copy=False)
        distances, indices = faiss_index.search(batch_embs, RETRIEVAL_TOP_K)

        for i in range(end - start):
            cid = bdc_ids[start + i]
            all_bdc_retrieval[cid] = list(zip(
                [psc_ids[idx] for idx in indices[i]],
                [float(d)     for d   in distances[i]]
            ))

        if (batch_num + 1) % CHECKPOINT_EVERY_N == 0:
            atomic_pickle_dump(all_bdc_retrieval, CHECKPOINT_PKL)

    atomic_pickle_dump(all_bdc_retrieval, ALL_BDC_RETRIEVAL_PKL)
    if CHECKPOINT_PKL.exists():
        CHECKPOINT_PKL.unlink()
    print(f'Saved → {ALL_BDC_RETRIEVAL_PKL}')

Loaded full retrieval: 961,431 entries


In [9]:
# Extract validation query retrieval lists
retrieval_results = {
    qid: all_bdc_retrieval[qid]
    for qid in query_ids
    if qid in all_bdc_retrieval
}
missing = set(query_ids) - set(retrieval_results.keys())
if missing:
    print(f'WARNING: {len(missing)} queries missing — {list(missing)[:5]}')
else:
    print(f'All {len(retrieval_results)} validation query retrievals loaded.')

sample_qid = query_ids[0]
print(f'\nTop-3 for {sample_qid}:')
for cid, score in retrieval_results[sample_qid][:3]:
    print(f'  {score:.4f}  {cid}')

All 100 validation query retrievals loaded.

Top-3 for 10015_sent_188_190:
  0.8364  022_Hieronymus-Stridonensis_Epistolae_window_755
  0.8268  035_Augustinus-Hipponensis_In-Joannis-evangelium-tractatus-CXXIV_window_2094
  0.8254  024_Hieronymus-Stridonensis_Commentaria-in-Isaiam_window_1794


## 4. Helper functions

In [10]:
def letter_id_from_chunk_id(chunk_id):
    """'10015_sent_0' → 'file10015'"""
    return 'file' + chunk_id.split('_')[0]


def rrf_score(rank, k=RRF_K):
    return 1.0 / (k + rank)


def reciprocal_rank_fusion(neighbour_lists, baseline_list, k=RRF_K):
    """
    Merge neighbour PSC lists with baseline retrieval list via RRF.
    Baseline contributes by rank; each neighbour list contributes independently.
    A PSC chunk appearing in multiple neighbour lists accumulates higher fused score.
    Returns [(psc_id, rrf_score), ...] sorted descending.
    """
    scores = defaultdict(float)
    for rank, (cid, _) in enumerate(baseline_list, start=1):
        scores[cid] += rrf_score(rank, k)
    for nlist in neighbour_lists:
        for rank, (cid, _) in enumerate(nlist, start=1):
            scores[cid] += rrf_score(rank, k)
    return sorted(scores.items(), key=lambda x: x[1], reverse=True)


def get_neighbours(G, letter_id, n):
    """Return up to n graph-adjacent letter_ids sorted by date_min."""
    if letter_id not in G:
        return []
    neighbours = sorted(G[letter_id].items(), key=lambda x: x[1].get('date_min', ''))
    return [nbr for nbr, _ in neighbours[:n]]


# Build letter_id → [chunk_ids] mapping for full BDC
letter_to_chunks = defaultdict(list)
for cid in bdc_ids:
    letter_to_chunks[letter_id_from_chunk_id(cid)].append(cid)
print(f'Letter → chunks mapping: {len(letter_to_chunks):,} letters')

# Graph connectivity stats for validation queries
val_letter_ids  = [letter_id_from_chunk_id(qid) for qid in query_ids]
connected_lids  = [lid for lid in val_letter_ids if lid in G and G.degree(lid) > 0]
isolated_lids   = [lid for lid in val_letter_ids if lid not in connected_lids]
print(f'Validation letters in graph with neighbours : {len(connected_lids)}')
print(f'Validation letters isolated / not in graph  : {len(isolated_lids)}')

Letter → chunks mapping: 11,858 letters
Validation letters in graph with neighbours : 49
Validation letters isolated / not in graph  : 51


In [11]:
def evaluate_stratified(results, validation_set, k_values=[5, 10, 20, 50, 100]):
    """
    Compute Recall@k and MRR@k stratified by:
      all | connected | isolated | explicit | implicit |
      connected_explicit | connected_implicit
    """
    entries = validation_set['ground_truth']

    connected_qids = set(
        e['query_chunk_id'] for e in entries
        if letter_id_from_chunk_id(e['query_chunk_id']) in G
        and G.degree(letter_id_from_chunk_id(e['query_chunk_id'])) > 0
    )

    def _compute(subset):
        metrics = {k: {'recall': [], 'mrr': []} for k in k_values}
        for entry in subset:
            qid    = entry['query_chunk_id']
            gt_ids = set(entry['relevant_chunks'])
            if qid not in results:
                continue
            retrieved = [cid for cid, _ in results[qid]]
            for k in k_values:
                top_k = retrieved[:k]
                metrics[k]['recall'].append(1.0 if any(c in gt_ids for c in top_k) else 0.0)
                mrr = 0.0
                for rank, cid in enumerate(top_k, 1):
                    if cid in gt_ids:
                        mrr = 1.0 / rank
                        break
                metrics[k]['mrr'].append(mrr)
        summary = {}
        for k in k_values:
            r = metrics[k]['recall']
            m = metrics[k]['mrr']
            summary[f'Recall@{k}'] = round(sum(r) / len(r), 4) if r else None
            summary[f'MRR@{k}']    = round(sum(m) / len(m), 4) if m else None
        return summary

    strata = {
        'all'               : entries,
        'connected'         : [e for e in entries if e['query_chunk_id'] in connected_qids],
        'isolated'          : [e for e in entries if e['query_chunk_id'] not in connected_qids],
        'explicit'          : [e for e in entries if e['reference_type'] == 'explicit'],
        'implicit'          : [e for e in entries if e['reference_type'] == 'implicit'],
        'connected_explicit': [e for e in entries if e['query_chunk_id'] in connected_qids and e['reference_type'] == 'explicit'],
        'connected_implicit': [e for e in entries if e['query_chunk_id'] in connected_qids and e['reference_type'] == 'implicit'],
    }

    return {stratum: _compute(subset) for stratum, subset in strata.items()}


# Baseline metrics
baseline_strat = evaluate_stratified(retrieval_results, validation_set)
print('=== Baseline (bi-encoder only) ===')
for stratum, m in baseline_strat.items():
    print(f'  {stratum:<25} Recall@100={m["Recall@100"]}  MRR@20={m["MRR@20"]}')

=== Baseline (bi-encoder only) ===
  all                       Recall@100=0.47  MRR@20=0.2689
  connected                 Recall@100=0.4898  MRR@20=0.2769
  isolated                  Recall@100=0.451  MRR@20=0.2612
  explicit                  Recall@100=0.8  MRR@20=0.5203
  implicit                  Recall@100=0.14  MRR@20=0.0176
  connected_explicit        Recall@100=0.7692  MRR@20=0.495
  connected_implicit        Recall@100=0.1739  MRR@20=0.0304


## 5. Stage 2 — Graph-neighbourhood pool expansion + RRF

Qualifying neighbour chunks (cosine similarity to query chunk ≥ `SIM_THRESHOLD`) contribute
their top-N PSC candidates to the query's candidate pool via union, potentially surfacing
PSC chunks the bi-encoder missed. Baseline and neighbour lists are merged via RRF (k=60).

In [12]:
def run_graph_expansion(retrieval_results, G, letter_to_chunks, all_bdc_retrieval,
                        n_neighbours, sim_threshold, top_n_from_neighbour=20):
    """
    Graph-neighbourhood candidate pool expansion.

    For each query chunk:
      1. Find n temporally closest graph-adjacent neighbour letters.
      2. For each neighbour chunk with cosine_sim(query, neighbour) >= sim_threshold,
         take its top-N PSC retrieval candidates.
      3. Merge all neighbour PSC lists + baseline via RRF.

    Returns:
        expanded_results : {query_chunk_id: [(psc_chunk_id, rrf_score), ...]}
        stats            : dict with diagnostic counts
    """
    expanded_results = {}
    isolated_count   = 0
    no_qualifying    = 0
    expansion_counts = []

    for query_chunk_id, baseline in tqdm(retrieval_results.items(), desc='Graph expansion'):
        query_letter_id = letter_id_from_chunk_id(query_chunk_id)
        neighbour_lids  = get_neighbours(G, query_letter_id, n_neighbours)

        if not neighbour_lids:
            isolated_count += 1
            expanded_results[query_chunk_id] = baseline
            continue

        q_idx = bdc_id_to_idx.get(query_chunk_id)
        if q_idx is None:
            expanded_results[query_chunk_id] = baseline
            continue
        q_emb = bdc_embeddings[q_idx]  # L2-normalised → dot = cosine sim

        baseline_ids        = set(cid for cid, _ in baseline)
        neighbour_psc_lists = []

        for nbr_lid in neighbour_lids:
            for nbr_chunk_id in letter_to_chunks.get(nbr_lid, []):
                nbr_idx = bdc_id_to_idx.get(nbr_chunk_id)
                if nbr_idx is None:
                    continue

                # Qualifying check: semantic proximity between query and neighbour chunk
                query_to_nbr_sim = float(np.dot(q_emb, bdc_embeddings[nbr_idx]))
                if query_to_nbr_sim < sim_threshold:
                    continue

                nbr_psc_list = all_bdc_retrieval.get(nbr_chunk_id)
                if not nbr_psc_list:
                    continue

                neighbour_psc_lists.append(nbr_psc_list[:top_n_from_neighbour])

        if not neighbour_psc_lists:
            no_qualifying += 1
            expanded_results[query_chunk_id] = baseline
            continue

        neighbour_ids  = set(cid for plist in neighbour_psc_lists for cid, _ in plist)
        new_candidates = neighbour_ids - baseline_ids
        expansion_counts.append(len(new_candidates))

        expanded_results[query_chunk_id] = reciprocal_rank_fusion(
            neighbour_psc_lists, baseline, k=RRF_K
        )

    stats = {
        'isolated'     : isolated_count,
        'no_qualifying': no_qualifying,
        'expanded'     : len(expansion_counts),
        'avg_new_cands': round(sum(expansion_counts) / len(expansion_counts), 1) if expansion_counts else 0,
        'max_new_cands': max(expansion_counts) if expansion_counts else 0,
    }

    print(f'Done.  Isolated: {isolated_count}  |  No qualifying: {no_qualifying}  |  '
          f'Expanded: {stats["expanded"]}  |  Avg new cands: {stats["avg_new_cands"]}')
    return expanded_results, stats

## 6. Hyperparameter sweep


In [13]:
N_VALUES   = [1, 2, 3, 5, 10]
T_VALUES   = [0.6, 0.7, 0.8, 0.9]
TOP_VALUES = [5, 10, 20, 30]

sweep_rows = []

for n, t, top_n in itertools.product(N_VALUES, T_VALUES, TOP_VALUES):
    fused, stats = run_graph_expansion(
        retrieval_results=retrieval_results,
        G=G,
        letter_to_chunks=letter_to_chunks,
        all_bdc_retrieval=all_bdc_retrieval,
        n_neighbours=n,
        sim_threshold=t,
        top_n_from_neighbour=top_n,
    )
    strat = evaluate_stratified(fused, validation_set)

    sweep_rows.append({
        'n': n, 't': t, 'top_n': top_n,
        'avg_new'       : stats['avg_new_cands'],
        'R@100_all'     : strat['all']['Recall@100'],
        'MRR@20_all'    : strat['all']['MRR@20'],
        'R@100_conn'    : strat['connected']['Recall@100'],
        'MRR@20_conn'   : strat['connected']['MRR@20'],
        'R@100_expl'    : strat['explicit']['Recall@100'],
        'R@100_impl'    : strat['implicit']['Recall@100'],
        'R@100_conn_ex' : strat['connected_explicit']['Recall@100'],
        'R@100_conn_im' : strat['connected_implicit']['Recall@100'],
    })

sweep_df = pd.DataFrame(sweep_rows)
print(sweep_df.to_string(index=False))

print('\n=== Baselines ===')
for stratum, m in baseline_strat.items():
    print(f'  {stratum:<25} Recall@100={m["Recall@100"]}  MRR@20={m["MRR@20"]}')

best = sweep_df.loc[sweep_df['R@100_conn'].idxmax()]
print(f'\nBest by connected Recall@100: n={best["n"]}, t={best["t"]}, top_n={best["top_n"]}')
print(f'  R@100_conn={best["R@100_conn"]}  MRR@20_conn={best["MRR@20_conn"]}  avg_new={best["avg_new"]}')

Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

Done.  Isolated: 51  |  No qualifying: 4  |  Expanded: 45  |  Avg new cands: 491.5


Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

Done.  Isolated: 51  |  No qualifying: 4  |  Expanded: 45  |  Avg new cands: 920.1


Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

...

Done.  Isolated: 51  |  No qualifying: 48  |  Expanded: 1  |  Avg new cands: 13.0


Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

Done.  Isolated: 51  |  No qualifying: 48  |  Expanded: 1  |  Avg new cands: 25.0


Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

Done.  Isolated: 51  |  No qualifying: 48  |  Expanded: 1  |  Avg new cands: 39.0
 n   t  top_n  avg_new  R@100_all  MRR@20_all  R@100_conn  MRR@20_conn  R@100_expl  R@100_impl  R@100_conn_ex  R@100_conn_im
 1 0.6      5    491.5       0.36      0.1452      0.2653       0.0244        0.64        0.08         0.4615         0.0435
 1 0.6     10    920.1       0.33      0.1438      0.2041       0.0217        0.58        0.08         0.3462         0.0435
 1 0.6     20   1702.6       0.29      0.1432      0.1224       0.0204        0.52        0.06         0.2308         0.0000
 1 0.6     30   2399.0       0.26      0.1432      0.0612       0.0204        0.46        0.06         0.1154         0.0000
 1 0.7      5    354.0       0.38      0.1466      0.3061       0.0274        0.68        0.08         0.5385         0.0435
 1 0.7     10    668.2       0.34      0.1450      0.2245       0.0242        0.60        0.08         0.3846         0.0435
 1 0.7     20   1244.5       0.31      0.14

## 7. Stage 3 — Cross-encoder reranking


In [14]:
print(f'Loading cross-encoder: {CROSS_ENCODER_MODEL}')
reranker = CrossEncoder(
    CROSS_ENCODER_MODEL,
    device=DEVICE,
    max_length=512,
    token = token,
    tokenizer_kwargs={'truncation': True, 'max_length': 512},
)
dummy       = reranker.predict([('test', 'test')], show_progress_bar=False)
dummy_t     = torch.tensor(dummy)
USE_SOFTMAX = dummy_t.ndim == 2 and dummy_t.shape[1] > 1
print(f'Output: {"softmax 2-class" if USE_SOFTMAX else "sigmoid 1-class"}')

The CrossEncoder `tokenizer_kwargs` argument was renamed and is now deprecated. Please use `processor_kwargs` instead.


Loading cross-encoder: julian-schelb/PhilBerta-class-latin-intertext-v1


config.json:   0%|          | 0.00/700 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/541M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Output: softmax 2-class


In [16]:
def rerank(results, desc='Reranking'):
    """Rerank a retrieval results dict with the PhilBERTa cross-encoder."""
    reranked = {}
    for query_chunk_id, cand_list in tqdm(results.items(), desc=desc):
        query_text = bdc_text_lookup.get(query_chunk_id, '.') or '.'
        candidates = cand_list[:RETRIEVAL_TOP_K]
        if not candidates:
            reranked[query_chunk_id] = []
            continue
        pairs = [
            (query_text, psc_text_lookup.get(cid, '.') or '.')
            for cid, _ in candidates
        ]
        raw_scores = reranker.predict(pairs, batch_size=RERANK_BATCH, show_progress_bar=False)
        raw_tensor = torch.tensor(raw_scores)
        probs = (
            F.softmax(raw_tensor, dim=1)[:, 1].numpy()
            if USE_SOFTMAX
            else torch.sigmoid(raw_tensor).numpy()
        )
        ranked = sorted(
            zip([cid for cid, _ in candidates], probs),
            key=lambda x: x[1], reverse=True
        )
        reranked[query_chunk_id] = ranked[:FINAL_K]
    return reranked

In [17]:
# Best expansion config from sweep
best_expanded, best_stats = run_graph_expansion(
    retrieval_results=retrieval_results,
    G=G,
    letter_to_chunks=letter_to_chunks,
    all_bdc_retrieval=all_bdc_retrieval,
    n_neighbours=int(best['n']),
    sim_threshold=float(best['t']),
    top_n_from_neighbour=int(best['top_n']),
)
print(f'Avg new candidates: {best_stats["avg_new_cands"]}')

Graph expansion:   0%|          | 0/100 [00:00<?, ?it/s]

Done.  Isolated: 51  |  No qualifying: 35  |  Expanded: 14  |  Avg new cands: 82.3
Avg new candidates: 82.3


In [18]:
gc.collect()
if DEVICE == 'cuda':
    torch.cuda.empty_cache()

reranked_baseline = rerank(retrieval_results, desc='Reranking baseline')
reranked_expanded = rerank(best_expanded,     desc='Reranking expanded pool')
print(f'Done. {len(reranked_baseline)} baseline + {len(reranked_expanded)} expanded reranked.')

Reranking baseline:   0%|          | 0/100 [00:00<?, ?it/s]

Reranking expanded pool:   0%|          | 0/100 [00:00<?, ?it/s]

Done. 100 baseline + 100 expanded reranked.


## 8. Final results

In [19]:
strat_bi_enc   = evaluate_stratified(retrieval_results, validation_set)
strat_expanded = evaluate_stratified(best_expanded,     validation_set)
strat_ce_base  = evaluate_stratified(reranked_baseline, validation_set)
strat_ce_exp   = evaluate_stratified(reranked_expanded, validation_set)

strata_to_print = ['all', 'connected', 'isolated', 'explicit', 'implicit',
                   'connected_explicit', 'connected_implicit']

for stratum in strata_to_print:
    a = strat_bi_enc[stratum]
    b = strat_expanded[stratum]
    c = strat_ce_base[stratum]
    d = strat_ce_exp[stratum]
    print(f'\n── {stratum} ──')
    print(f'  {"Metric":<12} {"Bi-enc":>10} {"+Expansion":>12} {"+CE base":>10} {"+CE exp":>10}')
    for metric in ['Recall@20', 'Recall@50', 'Recall@100', 'MRR@20']:
        print(f'  {metric:<12} {str(a.get(metric,"-")):>10} {str(b.get(metric,"-")):>12} '
              f'{str(c.get(metric,"-")):>10} {str(d.get(metric,"-")):>10}')

print(f'\nConfig: n={best["n"]}, t={best["t"]}, top_n={best["top_n"]} | CE input capped at top-{RETRIEVAL_TOP_K}')


── all ──
  Metric           Bi-enc   +Expansion   +CE base    +CE exp
  Recall@20           0.4         0.39       0.43       0.43
  Recall@50          0.44         0.44       0.43       0.43
  Recall@100         0.47         0.48       0.43       0.43
  MRR@20           0.2689       0.2481     0.2985     0.3038

── connected ──
  Metric           Bi-enc   +Expansion   +CE base    +CE exp
  Recall@20        0.4286       0.4082      0.449      0.449
  Recall@50        0.4694       0.4694      0.449      0.449
  Recall@100       0.4898       0.5102      0.449      0.449
  MRR@20           0.2769       0.2344     0.3206     0.3313

── isolated ──
  Metric           Bi-enc   +Expansion   +CE base    +CE exp
  Recall@20        0.3725       0.3725     0.4118     0.4118
  Recall@50        0.4118       0.4118     0.4118     0.4118
  Recall@100        0.451        0.451     0.4118     0.4118
  MRR@20           0.2612       0.2612     0.2774     0.2774

── explicit ──
  Metric           Bi-enc

In [20]:
# Save final reranked results (eval-compatible JSON)
for tag, res in [('baseline_ce', reranked_baseline), ('expanded_ce', reranked_expanded)]:
    out = {
        'metadata': {
            'model_name'    : f'e5-graph-rrf-philberta-{tag}',
            'cross_encoder' : CROSS_ENCODER_MODEL,
            'n_neighbours'  : int(best['n']),
            'sim_threshold' : float(best['t']),
            'top_n_neighbour': int(best['top_n']),
            'rrf_k'         : RRF_K,
            'retrieval_top_k': RETRIEVAL_TOP_K,
            'reranked_top_k': FINAL_K,
            'total_queries' : len(res),
        },
        'retrieval_results': {
            qid: [
                {'candidate_id': cid, 'similarity': float(score), 'rank': rank}
                for rank, (cid, score) in enumerate(cands, 1)
            ]
            for qid, cands in res.items()
        }
    }
    path = OUTPUT_DIR / f'retrieval_results_{tag}_n{int(best["n"])}_t{best["t"]}.json'
    with open(path, 'w') as f:
        json.dump(out, f, indent=2)
    print(f'Saved → {path}')

Saved → retrieval_rq3/retrieval_results_baseline_ce_n1_t0.8.json
Saved → retrieval_rq3/retrieval_results_expanded_ce_n1_t0.8.json


## EXAMPLE

In [24]:
for i, item in enumerate(improved):
    entry    = item["entry"]
    qid      = entry["query_chunk_id"]
    gt_ids   = item["gt_ids"]
    lid      = letter_id_from_chunk_id(qid)

    print(f"\n{'='*70}")
    print(f"Query #{i+1}: {qid}  (letter: {lid})")
    print(f"Reference type: {entry['reference_type']}  |  Church father: {entry.get('church_fathers','?')}")
    print(f"\n── Query text ──")
    print(bdc_text_lookup.get(qid, "NOT FOUND"))

    print(f"\n── Ground truth PSC chunk(s) ──")
    for gt in gt_ids:
        print(f"  {gt}")
        print(f"  {psc_text_lookup.get(gt, 'NOT FOUND')[:300]}")

    print(f"\n── Graph neighbours of {lid} ──")
    nbrs = get_neighbours(G, lid, N_NEIGHBOURS)
    for nbr_lid, data in G[lid].items():
        nbr_node = G.nodes[nbr_lid]
        print(f"  {nbr_lid}  date_min={data.get('date_min','')}  "
              f"{nbr_node.get('sender_ref','')} → {nbr_node.get('recipient_ref','')}")

    # Which neighbour chunk brought the GT into the pool?
    print(f"\n── Qualifying neighbour chunks that retrieved GT ──")
    q_idx = bdc_id_to_idx.get(qid)
    q_emb = bdc_embeddings[q_idx]

    for nbr_lid, _ in G[lid].items():
        for nbr_chunk_id in letter_to_chunks.get(nbr_lid, []):
            nbr_idx = bdc_id_to_idx.get(nbr_chunk_id)
            if nbr_idx is None:
                continue
            sim = float(np.dot(q_emb, bdc_embeddings[nbr_idx]))
            if sim < SIM_THRESHOLD:
                continue
            nbr_psc_list = all_bdc_retrieval.get(nbr_chunk_id, [])
            nbr_psc_ids  = [cid for cid, _ in nbr_psc_list[:TOP_N_FROM_NEIGHBOUR]]
            shared_gt    = gt_ids & set(nbr_psc_ids)
            if shared_gt:
                print(f"\n  Neighbour chunk : {nbr_chunk_id}")
                print(f"  Sim to query    : {sim:.4f}")
                print(f"  Neighbour text  : {bdc_text_lookup.get(nbr_chunk_id,'')[:300]}")
                print(f"  GT found in neighbour's top-{TOP_N_FROM_NEIGHBOUR}: {shared_gt}")
                for gt in shared_gt:
                    rank_in_nbr = next(
                        (r+1 for r, (c,_) in enumerate(nbr_psc_list) if c == gt), None
                    )
                    print(f"    GT rank in neighbour list: {rank_in_nbr}")

    print(f"\n── Where GT appears in expanded list ──")
    for gt in gt_ids:
        if gt in [c for c in item["expanded_retrieved"][:100]]:
            rank = item["expanded_retrieved"].index(gt) + 1
            print(f"  {gt}  → rank {rank} in expanded list")
        else:
            print(f"  {gt}  → NOT in expanded top-100")


Query #1: 10518_sent_99  (letter: file10518)
Reference type: implicit  |  Church father: P18700

── Query text ──
Rursus , cum sacramenta sint verba visibilia , quae docent sensibiliter , nomenque ipsum , quorum symbola sunt , sortiantur , nihil offendit nos , cum audimus panem dici corpus Christi .

── Ground truth PSC chunk(s) ──
  035_Augustinus-Hipponensis_In-Joannis-evangelium-tractatus-CXXIV_window_2106
  ait, propter verbum quod locutus sum vobis; nisi quia et in aqua verbum mundat? Detrahe verbum, et quid est aqua nisi aqua? Accedit verbum ad elementum, et fit Sacramentum, etiam ipsum tanquam visibile verbum. Nam et hoc utique dixerat, quando pedes discipulis lavit: Qui lotus est, non indiget nisi 

── Graph neighbours of file10518 ──
  file10501  date_min=1534-10-28  p495 → p467
  file10506  date_min=1534-11-01  p467 → p495
  file10514  date_min=1534-11-22  p467 → p495
  file10530  date_min=1534-12-15  p467 → p495
  file10553  date_min=1534-12-15  p467 → p495

── Qualifying n